# Part 3: The Machine Learning Workflow
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: From question to deployed model

Sarah's manager, Priya, is now sold on ML. She asks Sarah to **own** the churn-prediction project from end to end:

> "Figure out which of our current customers will churn in the next 3 months. Whatever you build, I want to use it every Monday to generate a 'save-this-customer' list for the retention team."

This is no longer a one-off analysis. It's a **workflow**: a repeatable process from business question → data → model → action → monitoring.

**By the end of this section you will:**
- Walk through the 7 standard steps of an ML project
- Frame a business problem rigorously (the most-often-skipped step)
- Recognise what can go wrong at each step
- Apply the workflow to a new scenario you haven't seen before

> *Sarah Chen · Customer Experience Analyst · NorthStar Retail · January 2023 (Thursday morning).*
> Sarah's report is due Friday. Before she presents, she needs to answer the hard questions Priya and the finance team (including Marcus, the finance partner) will ask. This notebook walks through the full ML workflow — from framing to monitoring — using Sarah's project as the case.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings

# scikit-learn prints internal deprecation notes that are not errors.
# We silence them here so the output stays clean for learning.
warnings.filterwarnings("ignore")

np.random.seed(42)
print("✅ Ready!")

## The 7-step workflow

Every serious ML project follows these steps, in order:

```
1. Frame the problem                (business question, success metric, who acts on the output)
2. Collect the data                 (where does it live? is there enough?)
3. Clean and explore                (missing values, duplicates, bias checks)
4. Train the model                  (choose a method, fit it to the data)
5. Evaluate on unseen data          (does it generalise?)
6. Deploy                           (make it usable by people / systems)
7. Monitor                          (the world changes; the model decays)
```

**Surprising time split on a real project:**

| Phase | % of project time |
|---|---|
| Steps 1–3 (framing, collection, cleaning) | **60–70%** |
| Steps 4–5 (training, evaluation) | **10–20%** |
| Steps 6–7 (deployment, monitoring) | **20%** |

Beginners think it's the opposite. It isn't.

## Step 1 — Frame the problem (the make-or-break step)

Before any code, Sarah should write down four things:

1. **Question:** *Which current customers are likely to stop shopping with us in the next 3 months?*
2. **Who acts on the answer:** Retention team. Every Monday they receive a ranked list, pick the top 200 at-risk customers, and send them a 15%-off email.
3. **Success metric:** Total churn rate *among customers emailed* must drop by 20% within 6 months, compared to customers who were also at-risk but not emailed (a holdout group).
4. **Cost of being wrong:**
   - *False positive* (model flags a customer who wasn't really going to churn): send them a coupon. Costs ~$5 in margin.
   - *False negative* (model misses a customer who churns): lose their $200/year spend. Costs ~$200.

**False negatives are 40x more expensive than false positives.** This will affect what kind of model she builds.

### ⏸️ Pause and Predict

Before Sarah does anything else, she needs to ask her manager some uncomfortable questions.

**Name two things that are still unclear from the framing above** — questions Sarah should push back on with her manager.

### Sample answer

Things Sarah should pin down before writing a single line of code:

- **What exactly counts as "churn"?** No purchase in 3 months? No email opened in 6 months? Different definitions change which customers the model flags.
- **Does 'at-risk' include brand-new customers** who have barely engaged yet? If yes, the model will flag every newcomer.
- **Who decides the 'top 200' threshold** — Sarah or the retention team? And can that threshold change weekly?
- **Will the same customer get repeated coupons** if they keep appearing at-risk? How often?
- **Who owns evaluation 6 months from now** — is there a plan to compare emailed vs holdout groups?

**Pushing back on the ask is part of the job. A well-framed problem is worth more than a clever model.**

## Step 2 — Collect the data

Sarah goes to the data warehouse. She needs **two kinds of data**:

- **Features:** things known about a customer *before* the churn event (demographics, past purchases, recency, frequency, support tickets...).
- **Labels:** did this customer churn, yes/no, in the following 3 months?

She can build labels from historical data: for each customer's state as of 6 months ago, did they churn in the following 3 months?

In [ ]:
# Simulate the data Sarah pulls from the warehouse
# Features: tenure (months), monthly_spend, days_since_last_visit, email_opens_last_30d
n = 1000

df = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "tenure_months": np.random.randint(1, 60, n),
    "monthly_spend": np.random.gamma(2, 40, n).round(2),
    "days_since_last_visit": np.random.exponential(30, n).astype(int),
    "email_opens_last_30d": np.random.poisson(3, n),
})

# Fake churn label: customers with very-low spend OR very-stale visits churn more
churn_score = (
    -0.03 * df["monthly_spend"]
    + 0.04 * df["days_since_last_visit"]
    - 0.2 * df["email_opens_last_30d"]
    - 0.02 * df["tenure_months"]
)
churn_prob = 1 / (1 + np.exp(-churn_score))
df["churned"] = (np.random.random(n) < churn_prob).astype(int)

print(f"Total customers: {len(df)}")
print(f"Churn rate:      {df['churned'].mean():.1%}")
df.head()

## Step 3 — Clean and explore

Before training, Sarah looks at the data. *Really* looks.

In [ ]:
print("--- Summary statistics ---")
print(df.describe().round(2))

print("\n--- Missing values per column ---")
print(df.isnull().sum())

print("\n--- Churn rate by tenure bracket ---")
df["tenure_bracket"] = pd.cut(df["tenure_months"], bins=[0, 12, 24, 60], labels=["<1yr", "1-2yr", "2yr+"])
print(df.groupby("tenure_bracket", observed=True)["churned"].mean().round(3))

### 💡 What do you notice?

- No missing values in this (synthetic) dataset — **in real data, expect to spend hours here.**
- The churn rate is highest in the **"less than 1 year" tenure bracket.** This aligns with common retail experience: new customers are most fragile.
- Before training anything, Sarah now has a **hypothesis** about what matters. The model will test it.

## Step 4 — Train a simple model

Sarah starts with the simplest reasonable model: **logistic regression**. Not because it's the best — because it's fast, interpretable, and a baseline.

In [ ]:
features = ["tenure_months", "monthly_spend", "days_since_last_visit", "email_opens_last_30d"]
X = df[features].values
y = df["churned"].values

# Split into train and test BEFORE fitting anything
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"Training set: {len(X_train)}  |  Test set: {len(X_test)}")

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("\n✅ Model trained on training set.")

# What did it learn?
print("\nFeature influence on churn (positive = increases churn risk):")
for f, coef in zip(features, model.coef_[0]):
    arrow = "↑" if coef > 0 else "↓"
    print(f"  {f:>25}: {coef:+.3f}  {arrow}")

## Step 5 — Evaluate on unseen data

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("--- Classification report ---")
print(classification_report(y_test, y_pred, target_names=["stays", "churns"]))

print("--- Confusion matrix ---")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=["true stays", "true churns"], columns=["pred stays", "pred churns"])
print(cm_df)

### 💡 What do you notice?

- The **confusion matrix** tells Sarah exactly where the model fails — including how many real churners the model missed (*false negatives*).
- Remember from framing: **false negatives cost 40x more than false positives.** She may want to adjust the prediction threshold to catch more churners even if it means more false positives.
- We'll formalise this trade-off in L03 (precision, recall, ROC curves). For now, just notice that *"accuracy"* alone doesn't capture what the business actually cares about.

### Adjusting the threshold for Sarah's business reality

In [ ]:
# Instead of the default 0.5 threshold, use 0.3 — catch more churners
threshold = 0.3
y_pred_low_threshold = (y_proba >= threshold).astype(int)

print(f"At threshold {threshold}:")
cm_low = confusion_matrix(y_test, y_pred_low_threshold)
cm_df_low = pd.DataFrame(cm_low, index=["true stays", "true churns"], columns=["pred stays", "pred churns"])
print(cm_df_low)
print(f"\nCustomers flagged for retention email: {y_pred_low_threshold.sum()} out of {len(y_pred_low_threshold)}")

### 💡 What do you notice?

- **Lowering the threshold catches more real churners** but flags more false alarms.
- Sarah doesn't need to tune the model to a single "best" threshold right now — she **tunes it to the business cost ratio** (40:1 in her case).
- This is *why* framing the cost of wrongness in Step 1 matters so much: it gives you the ratio to tune against here.

## Steps 6 & 7 — Deploy and Monitor (the hidden 40%)

Sarah's model now works on a notebook. But **nobody can use a notebook in their daily workflow.** Deployment typically means one of:

- A **batch pipeline** that runs every Monday, scores all customers, writes a CSV, and emails it to the retention team. (Probably what Sarah starts with.)
- A **real-time API** that the customer-service tool calls when a support agent is on a call — "is this customer high-churn-risk?"
- An **embedded dashboard** in the team's BI tool where a filter shows "top-200 at-risk this week."

Once deployed, **monitoring** is essential:
- **Performance metrics:** is the model's accuracy on last week's data still above threshold?
- **Drift metrics:** are the input distributions (monthly spend, recency) shifting compared to training data?
- **Business metrics:** is the emailed cohort actually churning less than the holdout?

**When any of these degrade, it's time to retrain.**

## 🧠 Apply the workflow to a new scenario

**A hospital wants to predict which admitted patients are at high risk of readmission within 30 days.**

For each of the 7 steps, write **one concrete action Sarah would take** for this project. Take 5 minutes.

**Example for step 1:** *"Meet with the readmissions team to understand what counts as 'high-risk' — top 10% of all admissions? Or anyone above a specific probability threshold? Who decides?"*

*Your answers:*

1. Frame:
2. Collect:
3. Clean/explore:
4. Train:
5. Evaluate:
6. Deploy:
7. Monitor:

### Sample answer

1. **Frame:** Meet with clinicians and operations. Define "high-risk" clearly. Who acts on the prediction — the discharge nurse? A care coordinator? What intervention do they take? Success metric: reduction in 30-day readmissions among the flagged group.
2. **Collect:** Pull 2+ years of admissions data with features known at discharge (age, comorbidities, length of stay, lab results, medications, discharge destination) and labels (did they come back within 30 days?).
3. **Clean/explore:** Check for missing values, duplicate admissions, coding inconsistencies. Look at readmission rates by ward, age bracket, diagnosis — form hypotheses before modelling.
4. **Train:** Start with logistic regression as a baseline. Try a random forest (L04) for comparison.
5. **Evaluate:** Use a test set held back by time (train on 2022–2023, test on first half of 2024). Check recall for high-risk cases specifically, not just overall accuracy.
6. **Deploy:** Batch score every patient at discharge, write result to the electronic health record as a flag the coordinator sees.
7. **Monitor:** Weekly tracking of model accuracy and input distributions. Retrain quarterly or whenever drift is detected.

## ✅ Section 3 Summary

| Step | What happens | Where beginners get stuck |
|---|---|---|
| **1. Frame** | Business question, success metric, cost of wrongness | Skipping it entirely |
| **2. Collect** | Gather features + labels | Assuming the data exists when it doesn't |
| **3. Clean/explore** | Fix missing, check for bias | Underestimating how long this takes |
| **4. Train** | Fit a model on training data | Training on all the data (no test split) |
| **5. Evaluate** | Measure on held-out data | Optimising for accuracy when the business wants recall |
| **6. Deploy** | Put the model where it gets used | Leaving the model stuck in a notebook |
| **7. Monitor** | Track performance over time | Assuming "trained once = good forever" |

**Key insight:**
> A beautifully trained model that solves the wrong problem is worth zero. The framing step (#1) is where senior practitioners earn their keep.

## 🏁 Module L01 complete — and the question Priya hasn't let go of

You've covered the three foundational sections:

- **Part 1:** What ML *is* — rules vs examples
- **Part 2:** The three categories — supervised, unsupervised, reinforcement
- **Part 3:** The 7-step workflow from question to deployed model

Sarah writes up the workflow and walks into Priya's office with the final numbers. Priya nods, then looks up:

> *"Sarah — the numbers look great. But **how sure are we**? The model put a label on every review and every customer. How do we know it isn't confidently wrong in ways we can't see from this deck?"*

Sarah doesn't have a clean answer yet. That question — *how sure are we?* — is the engine of **Lesson 2**. Come to L02 ready to help Sarah answer it.

---

**Next step → `notebooks/assignment.ipynb`.**
Apply the full ML workflow to a completely different domain: Tom Bradley at Lakeside Bank is trying to predict loan default. Work through the practice tiers and the assignment exercises before our next session.